# Surface Mesh Segmentation Test

This notebook tests the surface mesh segmentation functionality.

## Imports and boiler-plate code

In [ ]:
import unittest
import numpy as np
import sys
import os

# Add code directory to path to import pySHP modules
# Notebook is in: code/pySHP/tests/
# We need to add: code/ to the path
code_dir = r'C:\Users\Khaled Khairy\Dropbox\Projects\hot\Project_spherical_parameterization\code'
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

from pySHP.surface_mesh import surface_mesh
from pySHP.shp_surface import shp_surface
from pySHP.utils import readoff
from pySHP.level1.mesh_segmentation_rw import mesh_segmentation_rw
from pySHP.level1.patch_info_gen import patch_info_gen  # Restart kernel after editing this module

# PyVista: 'static' is most reliable (always shows). For interactive rotation, open the
# auto-exported HTML in a browser (plot_simplified_mesh exports by default).
import pyvista as pv
pv.set_jupyter_backend('static')
print("PyVista: static backend (reliable); HTML export for interactive 3D in browser")
# Before optimize: ensure pyacvd is available (optional check)
try:
    import pyacvd
    print("pyacvd is installed — optimize() will use CVT resampling.")
except ImportError:
    print("pyacvd not installed. Install with: pip install pyacvd")

## Load Mesh Data

In [ ]:
# Read OFF file
topic = 'test_set' #'scientific' #'misc_shapes'
file_name = 'mushroom_repaired_03.off' #'1dpx.off' #'planula_01.off'  #'test_mud_01.off' #'MaxPlankHead.off'
fn_shape = os.path.join('C:\\Users\\Khaled Khairy\\Dropbox\\Projects\\hot\\Project_spherical_parameterization\\code', 
'Matlab','shp_toolbox-main', 'shp_toolbox-main', 'test_data', 'off', 
topic,
file_name 
)
X, F = readoff(fn_shape)
m = surface_mesh(X, F)
m.disp()
m.plot()

## Mesh Preprocessing and Quality Check

In [ ]:
##KK optimize the mesh ---> this takes some time, likely produce 4x the number of vertices specified (check and simplify )
#m.optimize(target_vertices=1000, debug=True, max_iter=2)
#m.plot()
#m.info()
### blender
##script_path = m.export_blender("mushroom.blend", run_blender=True)

In [ ]:
## Mesh Preprocessing and Quality Check
m.repair_mesh()
components = m.find_disconnected_surfaces()
m.keep_largest_surface()
m.info()  # Check initial topology
m.check_mesh_integrity()  # Check for problems
m.print_mesh_quality()  # See triangle quality

####m.remesh_uniform(1000, n_iterations=10, smooth_iterations=5) # ---> don't use

###KK the "optimized" mesh likely has too many vertices/faces --> simplify here to get a high quality mesh with fewer faces --- takes a long time!!
#m.simplify_mesh(4000)
#fn_shape_out = os.path.join('C:\\Users\\Khaled Khairy\\Dropbox\\Projects\\hot\\Project_spherical_parameterization\\code', 
#'Matlab','shp_toolbox-main', 'shp_toolbox-main', 'test_data', 'off', topic,'mushroom_repaired_03.off' )
#m.write_off(fn_shape_out)
##m.remesh_curvature_adaptive(500, curvature_strength=1.0, n_iterations=5, smooth_iterations=5)

### Curvature-adaptive remeshing (optional): denser mesh in high-curvature regions,
### coarser in flat regions.  Improves downstream spherical parameterization by
### spreading curvature information more uniformly and helping segmentation
### naturally create patches aligned with curvature features.
### curvature_strength controls density ratio: 2.0 gives 9:1 density ratio between
### highest-curvature and flattest areas.  Increase for sharper features.
m.remesh_by_curvature(target_faces=len(m.F), curvature_strength=2.0)
m.plot_H()  # visualize mean curvature field after remeshing

m.disp()
m.plot()


## Fine Mesh Segmentation

Generate mesh segmentation using random walk method.

In [ ]:
# Generate mesh segmentation + curvature-aware decimated simplified mesh
# Uses half-edge collapse to decimate the original mesh while preserving:
#   - original vertex positions (no interpolation)
#   - high-curvature regions (more detail kept where curvature is high)
#   - key vertices (triple junctions, sentinels, centers are protected)
from pySHP.level1.build_pm_from_decimated_mesh import find_valid_segmentation_with_decimated_mesh

sigma = 1.0
curvature_weight = -1e-6
failure_log = os.path.join(os.getcwd(), 'segmentation_failures_decimated.jsonl')

result = find_valid_segmentation_with_decimated_mesh(
    m,
    nseeds_range=[15],              # fixed seed count
    min_neighbors=3,
    sig=sigma,
    curvature_weight_seg=curvature_weight,
    curvature_weight_dec=1.0,       # bias decimation toward keeping high-curvature vertices
    target_ratio=0.08,              # keep ~8% of original faces (~320 faces for a 4000-face mesh)
    verbose=True,
    plot_intermediate=True,
    allow_annular=True,
    failure_log_path=failure_log,
)

if result is None:
    raise RuntimeError("No valid segmentation + decimated mesh found. Check segmentation_failures_decimated.jsonl")

ms    = result['ms']
L     = result['L']
slix  = result['slix']
P     = result['P']
Pconn = result['Pconn']
m_seg = result['m_seg']
PM    = result['PM']

print(f"\nSuccess: nseeds={result['nseeds']}")
print(f"  Decimated mesh: {len(PM['pm'].X)} vertices, {len(PM['pm'].F)} faces")
print(f"  Patches: {PM['npatches']}")
print(f"  Key vertices in decimated mesh: {len(PM['Keyind'])}")
print(f"  Center vertices in decimated mesh: {len(PM['CVind'])}")


### Verify Segmentation Results

## Generate Patches and Simplified Mesh (made of key vertices and center vertices)

In [ ]:
# patch_info_gen is now integrated in find_valid_segmentation_with_simplified_mesh
# m_seg, PM come from result - no separate call needed
print(f"  Simplified mesh: {len(PM['pm'].X)} vertices, {len(PM['pm'].F)} faces")
print(f"  Patches: {len(P)}")

In [ ]:
# Full diagnostic of simplified mesh topology (right after patch_info_gen / simplified mesh generation)
import os
from pySHP.level1.diagnose_simplified_mesh import diagnose_simplified_mesh_full

if PM is not None and 'pm' in PM and PM['pm'] is not None:
    base = code_dir if 'code_dir' in dir() else os.getcwd()
    diag_file = os.path.join(base, 'pySHP', 'tests', 'simplified_mesh_diagnostic_full.txt')
    report = diagnose_simplified_mesh_full(PM, verbose=True, output_file=diag_file)
    print(f"\nDiagnostic written to: {diag_file}")
    if report['valid']:
        print("Simplified mesh is VALID: ready for bijective mapping to sphere.")
    else:
        print("Simplified mesh has issues. See report above for details.")
else:
    print("PM or PM['pm'] not found; run patch_info_gen first.")

### Visualize Simplified Mesh

In [ ]:
import pyvista as pv
pv.set_jupyter_backend('static')
# [1] Plot the simplified mesh
print("\n[1] Plotting simplified mesh...")
html_dir = os.path.join('C:\\Users\\Khaled Khairy\\Dropbox\\Projects\\hot\\Project_spherical_parameterization\\code', 
'pySHP','tests', 'simplified_patch_html')
plot_html = os.path.join(html_dir, 'simplified_mesh_plot.html')
surface_mesh.plot_simplified_mesh(PM, show_keys=True, show_cv=True,subplot_shape=(1, 2), export_html_path=plot_html)

### Verify Patch Edges in PM


In [ ]:
# [4] Show border vertices on the full mesh
print("\n[4] Plotting full mesh with border vertices...")
m_seg.plot_border()

## Spherical Parameterization of Simplified Mesh


In [ ]:
# Spherically parameterize simplified mesh (optimization_method 3: area + edge-length)
from pySHP.level1.set_edge_n_fine_vertices import set_edge_n_fine_vertices_from_PM

pm = PM['pm']
ms = surface_mesh(pm.X, pm.F)

# Set per-edge fine-vertex counts from PM (same border chains as patch parameterization)
ms = set_edge_n_fine_vertices_from_PM(ms, PM)
# start out with area minimization
ms.needs_mesh2sphere = 1
ms.optimization_method = 1   # area Newton steps
ms.newton_niter = 30
ms.newton_step = 0.1         # area steps
ms.newton_step_edge = 0.05   # edge-length steps
ms.bijective_plot_flag = 2
ms.map2sphere()
ms.plot_spherical_parameterization()

In [ ]:

# now minimize  edge length
# Optional: use relative error so all edges weighted equally (can help correlation)
ms.edge_length_relative_error = True   # try True if correlation stays negative
ms.optimization_method = 4
ms.prevent_flip = True
ms.newton_niter = 30
ms.newton_step_edge = 0.5
ms.map2sphere()
sp_path = os.path.join('C:\\Users\\Khaled Khairy\\Dropbox\\Projects\\hot\\Project_spherical_parameterization\\code', 
'pySHP','tests', 'simplified_patch_html',
'sp_path.html' 
)
ms.plot_spherical_parameterization(export_html_path=sp_path)



### Update PM with optimized simplified mesh values

In [ ]:
# Update PM['pm'] with the theta and phi values from the parameterized simplified mesh
# After successful spherical parameterization of ms, copy t and p to PM['pm']
print("="*60)
print("Updating PM['pm'] with parameterized theta and phi values")
print("="*60)

# Copy the parameterized values from ms to PM['pm']
PM['pm'].t = ms.t.copy()
PM['pm'].p = ms.p.copy()

print(f"Updated PM['pm'] with theta and phi values")
print(f"  PM['pm'] has {len(PM['pm'].X)} vertices")
print(f"  Theta range: [{PM['pm'].t.min():.4f}, {PM['pm'].t.max():.4f}]")
print(f"  Phi range: [{PM['pm'].p.min():.4f}, {PM['pm'].p.max():.4f}]")
print("\n" + "="*60)

## Patch-based fine-mesh spherical parameterization

### Parameterize the fine mesh patches using the simplified mesh parameterization

In [ ]:

import importlib, sys
_mod_name = 'pySHP.level1.parameterize_patches_cart'
if _mod_name in sys.modules:
    importlib.reload(sys.modules[_mod_name])
from pySHP.level1.parameterize_patches_cart import parameterize_patches_cart

print("="*60)
print("Parameterizing fine mesh patches")
print("="*60)
print(f"Number of patches to parameterize: {PM['npatches']}")

# Parameterize all patches
# This will project each fine mesh patch onto the sphere using the boundary
# conditions from the simplified mesh parameterization
PM = parameterize_patches_cart(PM, plot_flag=1)

print("Patch parameterization complete!")
print(f"Parameterized {PM['npatches']} patches")


### visualization for a specific patch

In [ ]:
# This shows: fine mesh, border vertices, chain vertices, sentinel vertices

print("="*60)
print("Patch Debug Visualization")
print("="*60)

# Select patch to visualize (change this to debug different patches)
patch_idx = 8  # Change this to visualize different patches (0-indexed)

if patch_idx >= PM['npatches']:
    print(f"Error: Patch index {patch_idx} is out of range (max: {PM['npatches']-1})")
else:
    print(f"Visualizing patch {patch_idx}")
    
    try:
        import pyvista as pv
        #pv.set_jupyter_backend('static')
        # Get the patch mesh
        patm = PM['P'][patch_idx][0]
        
        # Get border vertices
        border_vertices = np.where(patm.border_vertex)[0] if hasattr(patm, 'border_vertex') and patm.border_vertex is not None else np.array([])
        
        # Get chain vertices from edge_dat
        chain_vertices = set()
        if 'patch' in PM and patch_idx in PM.get('patch', {}):
            # Use refined edge chains
            for edge_chain in PM['patch'][patch_idx]['edge_dat']:
                chain_vertices.update(edge_chain.astype(int))
        else:
            # Use PM.edge_dat
            for eix in range(len(PM['Edges'])):
                if PM['Edges'][eix, 0] == patch_idx or PM['Edges'][eix, 1] == patch_idx:
                    if eix < len(PM['edge_dat']) and len(PM['edge_dat'][eix]) > 0:
                        chain_vertices.update(PM['edge_dat'][eix].astype(int))
        chain_vertices = np.array(list(chain_vertices))
        
        # Get sentinel vertices for this patch
        sentinel_vertices = set()
        for eix in range(len(PM['Edges'])):
            if PM['Edges'][eix, 0] == patch_idx or PM['Edges'][eix, 1] == patch_idx:
                if eix < len(PM['sentinels']):
                    s1 = int(PM['sentinels'][eix, 0])
                    s2 = int(PM['sentinels'][eix, 1])
                    if s1 >= 0 and s1 < len(m.X):
                        sentinel_vertices.add(s1)
                    if s2 >= 0 and s2 < len(m.X):
                        sentinel_vertices.add(s2)
        sentinel_vertices = np.array(list(sentinel_vertices))
        
        # Create subplot layout: 2x3 grid (5 plots: 2x2 + 1 extra)
        plotter = pv.Plotter(shape=(2, 3))
        
        # ========== Plot 1: Fine mesh of the patch ==========
        plotter.subplot(0, 0)
        # Create mesh from patch
        num_faces = patm.F.shape[0]
        faces_with_n_vertices = np.hstack((np.full((num_faces, 1), 3), patm.F))
        cells = faces_with_n_vertices.flatten()
        mesh_pv = pv.PolyData(patm.X, cells)
        plotter.add_mesh(mesh_pv, color='lightblue', show_edges=True, edge_color='black', line_width=0.5)
        plotter.add_text(f'Patch {patch_idx}: Fine Mesh\n({len(patm.X)} vertices, {len(patm.F)} faces)', font_size=10)
        plotter.background_color = 'white'
        
        # ========== Plot 2: Patch mesh with border vertices marked ==========
        plotter.subplot(0, 1)
        plotter.add_mesh(mesh_pv, color='lightblue', show_edges=True, edge_color='black', line_width=0.5)
        if len(border_vertices) > 0:
            border_points = pv.PolyData(patm.X[border_vertices])
            plotter.add_mesh(border_points, color='yellow', point_size=15,
                           render_points_as_spheres=True, label=f'Border ({len(border_vertices)})')
        plotter.add_text(f'Patch {patch_idx}: Border Vertices\n(Yellow spheres)', font_size=10)
        plotter.background_color = 'white'
        plotter.add_legend()
        
        # ========== Plot 3: Patch mesh with chain vertices marked ==========
        plotter.subplot(1, 0)
        plotter.add_mesh(mesh_pv, color='lightblue', show_edges=True, edge_color='black', line_width=0.5)
        if len(chain_vertices) > 0:
            # Filter chain vertices that are in the patch
            chain_in_patch = chain_vertices[chain_vertices < len(patm.X)]
            if len(chain_in_patch) > 0:
                chain_points = pv.PolyData(patm.X[chain_in_patch])
                plotter.add_mesh(chain_points, color='orange', point_size=12,
                               render_points_as_spheres=True, label=f'Chain ({len(chain_in_patch)})')
        plotter.add_text(f'Patch {patch_idx}: Chain Vertices\n(Orange spheres)', font_size=10)
        plotter.background_color = 'white'
        plotter.add_legend()
        
        # ========== Plot 4: Full object mesh with patch border and sentinels ==========
        plotter.subplot(1, 1)
        # Create full mesh
        num_faces_full = m.F.shape[0]
        faces_with_n_vertices_full = np.hstack((np.full((num_faces_full, 1), 3), m.F))
        cells_full = faces_with_n_vertices_full.flatten()
        mesh_full = pv.PolyData(m.X, cells_full)
        
        # Color only the patch faces
        face_colors = np.ones((len(m.F), 3)) * 0.9  # Light gray for all faces
        if hasattr(m, 'face_labels') and m.face_labels is not None:
            # Get faces belonging to this patch
            unique_labels = np.unique(m.face_labels)
            if patch_idx < len(unique_labels):
                patch_label = unique_labels[patch_idx]
                patch_faces = np.where(m.face_labels == patch_label)[0]
                face_colors[patch_faces] = [0.5, 0.8, 1.0]  # Light blue for patch faces
        
        mesh_full['face_colors'] = face_colors
        plotter.add_mesh(mesh_full, scalars='face_colors', rgb=True, show_edges=True, 
                        edge_color='black', line_width=0.3, opacity=0.7)
        
        # Mark border vertices of this patch on the full mesh
        if len(border_vertices) > 0:
            # Map border vertices from patch to full mesh (they use same indices)
            border_points_full = pv.PolyData(m.X[border_vertices])
            plotter.add_mesh(border_points_full, color='yellow', point_size=12,
                           render_points_as_spheres=True, label=f'Border ({len(border_vertices)})')
        
        # Mark sentinel vertices as red spheres
        if len(sentinel_vertices) > 0:
            sentinel_points = pv.PolyData(m.X[sentinel_vertices])
            plotter.add_mesh(sentinel_points, color='red', point_size=20,
                           render_points_as_spheres=True, label=f'Sentinels ({len(sentinel_vertices)})')
        
        plotter.add_text(f'Full Mesh: Patch {patch_idx} Highlighted\n(Yellow=border, Red=sentinels)', font_size=10)
        plotter.background_color = 'white'
        plotter.add_legend()
        
        # ========== Plot 5: Spherical parameterization of the patch ==========
        plotter.subplot(1, 2)
        
        # Check if patch has t and p values
        if patm.t is not None and patm.p is not None and np.any(patm.t != 0) and np.any(patm.p != 0):
            # Convert spherical coordinates to Cartesian on unit sphere
            # Use the same import as the rest of the codebase
            from pySHP.utils import kk_sph2cart
            u, v, w = kk_sph2cart(patm.t, patm.p, np.ones(len(patm.p)))
            X_sph = np.column_stack([u, v, w])
            
            # Create mesh on sphere
            num_faces_sph = patm.F.shape[0]
            faces_with_n_vertices_sph = np.hstack((np.full((num_faces_sph, 1), 3), patm.F))
            cells_sph = faces_with_n_vertices_sph.flatten()
            mesh_sph = pv.PolyData(X_sph, cells_sph)
            
            # Add reference sphere (semi-transparent)
            sphere = pv.Sphere(radius=0.98, theta_resolution=30, phi_resolution=30)
            plotter.add_mesh(sphere, color='cyan', opacity=0.3, show_edges=False)
            
            # Plot the parameterized patch on sphere
            plotter.add_mesh(mesh_sph, color='lightblue', show_edges=True, 
                           edge_color='black', line_width=0.5, opacity=0.9)
            
            # Mark border vertices on sphere
            if len(border_vertices) > 0:
                border_vertices_valid = border_vertices[(patm.t[border_vertices] != 0) & (patm.p[border_vertices] != 0)]
                if len(border_vertices_valid) > 0:
                    border_points_sph = pv.PolyData(X_sph[border_vertices_valid])
                    plotter.add_mesh(border_points_sph, color='yellow', point_size=12,
                                   render_points_as_spheres=True, label=f'Border ({len(border_vertices_valid)})')
            
            plotter.add_text(f'Patch {patch_idx}: Spherical Parameterization\n(On unit sphere)', font_size=10)
            plotter.background_color = 'black'
            plotter.add_legend()
        else:
            plotter.add_text(f'Patch {patch_idx}: No Parameterization\n(t and p values not available)', font_size=10)
            plotter.background_color = 'white'
        
        # Show the plot (static backend)
        plotter.show()
        
        # === HTML export: interactive sphere view for this patch ===
        try:
            import os as _os
            html_dir = _os.path.join(code_dir, 'pySHP', 'tests', 'patch_sphere_html')
            _os.makedirs(html_dir, exist_ok=True)
            html_path = _os.path.join(html_dir, f'patch_{patch_idx}_sphere.html')
            p_html = pv.Plotter()
            sphere_ref = pv.Sphere(radius=0.98, theta_resolution=30, phi_resolution=30)
            p_html.add_mesh(sphere_ref, color='cyan', opacity=0.15, show_edges=False)
            if patm.t is not None and patm.p is not None and np.any(patm.t != 0):
                from pySHP.utils import kk_sph2cart
                u_h, v_h, w_h = kk_sph2cart(patm.t, patm.p, np.ones(len(patm.p)))
                X_sph_h = np.column_stack([u_h, v_h, w_h])
                nf_h = patm.F.shape[0]
                cells_h = np.hstack((np.full((nf_h, 1), 3), patm.F)).flatten()
                mesh_sph_h = pv.PolyData(X_sph_h, cells_h)
                p_html.add_mesh(mesh_sph_h, color='lightblue', show_edges=True,
                                edge_color='black', line_width=0.5, opacity=0.9)
                if len(border_vertices) > 0:
                    bv_valid = border_vertices[
                        (patm.t[border_vertices] != 0) | (patm.p[border_vertices] != 0)]
                    if len(bv_valid) > 0:
                        p_html.add_mesh(pv.PolyData(X_sph_h[bv_valid]),
                                        color='yellow', point_size=8,
                                        render_points_as_spheres=True)
                if len(sentinel_vertices) > 0:
                    sv_valid = sentinel_vertices[sentinel_vertices < len(X_sph_h)]
                    if len(sv_valid) > 0:
                        p_html.add_mesh(pv.PolyData(X_sph_h[sv_valid]),
                                        color='red', point_size=14,
                                        render_points_as_spheres=True)
            p_html.add_text(f'Patch {patch_idx} on unit sphere', font_size=12)
            p_html.background_color = 'white'
            p_html.export_html(html_path)
            p_html.close()
            print(f'HTML sphere view exported: {html_path}')
        except Exception as e_html:
            print(f'HTML export failed: {e_html}')
        
        # Print diagnostic information
        print(f"\nPatch {patch_idx} Diagnostics:")
        print(f"  Patch vertices: {len(patm.X)}")
        print(f"  Patch faces: {len(patm.F)}")
        print(f"  Border vertices: {len(border_vertices)}")
        print(f"  Chain vertices: {len(chain_vertices)}")
        print(f"  Sentinel vertices: {len(sentinel_vertices)}")
        if len(sentinel_vertices) > 0:
            print(f"    Sentinel indices: {sentinel_vertices}")
        
        # Check overlap
        border_set = set(border_vertices)
        chain_set = set(chain_vertices[chain_vertices < len(patm.X)])
        overlap = border_set.intersection(chain_set)
        border_only = border_set - chain_set
        chain_only = chain_set - border_set
        
        print(f"\n  Border-Chain Overlap:")
        print(f"    Overlap: {len(overlap)} vertices")
        print(f"    Border-only: {len(border_only)} vertices")
        print(f"    Chain-only: {len(chain_only)} vertices")
        
        # Spherical parameterization diagnostics
        if patm.t is not None and patm.p is not None:
            valid_tp = (patm.t != 0) & (patm.p != 0)
            num_valid = np.sum(valid_tp)
            print(f"\n  Spherical Parameterization:")
            print(f"    Vertices with valid t,p: {num_valid} / {len(patm.t)}")
            if num_valid > 0:
                print(f"    Theta range: [{patm.t[valid_tp].min():.4f}, {patm.t[valid_tp].max():.4f}]")
                print(f"    Phi range: [{patm.p[valid_tp].min():.4f}, {patm.p[valid_tp].max():.4f}]")
        else:
            print(f"\n  Spherical Parameterization: Not available (t or p is None)")
        
    except ImportError:
        print("PyVista not available for visualization")
    except Exception as e:
        print(f"Error creating visualization: {e}")
        import traceback
        traceback.print_exc()

# === Export ALL patches as interactive HTML sphere views ===
print("\n" + "="*60)
print("Exporting ALL patches as HTML sphere views...")
print("="*60)
try:
    import os as _os
    html_dir = _os.path.join(code_dir, 'pySHP', 'tests', 'patch_sphere_html')
    _os.makedirs(html_dir, exist_ok=True)
    from pySHP.utils import kk_sph2cart
    for _pix in range(PM['npatches']):
        _patm = PM['P'][_pix][0]
        _html_path = _os.path.join(html_dir, f'patch_{_pix}_sphere.html')
        _p = pv.Plotter()
        _sphere = pv.Sphere(radius=0.98, theta_resolution=30, phi_resolution=30)
        _p.add_mesh(_sphere, color='cyan', opacity=0.15, show_edges=False)
        if _patm.t is not None and _patm.p is not None and np.any(_patm.t != 0):
            _u, _v, _w = kk_sph2cart(_patm.t, _patm.p, np.ones(len(_patm.p)))
            _Xsph = np.column_stack([_u, _v, _w])
            _nf = _patm.F.shape[0]
            _cells = np.hstack((np.full((_nf, 1), 3), _patm.F)).flatten()
            _mesh_sph = pv.PolyData(_Xsph, _cells)
            _p.add_mesh(_mesh_sph, color='lightblue', show_edges=True,
                        edge_color='black', line_width=0.5, opacity=0.9)
            # Mark border vertices
            if hasattr(_patm, 'border_vertex') and _patm.border_vertex is not None:
                _bv = np.where(_patm.border_vertex.astype(bool))[0]
                _bv = _bv[(_patm.t[_bv] != 0) | (_patm.p[_bv] != 0)]
                if len(_bv) > 0:
                    _p.add_mesh(pv.PolyData(_Xsph[_bv]), color='yellow',
                                point_size=6, render_points_as_spheres=True)
        _p.add_text(f'Patch {_pix} on sphere ({_patm.F.shape[0]} faces)', font_size=12)
        _p.background_color = 'white'
        _p.export_html(_html_path)
        _p.close()
        print(f'  Patch {_pix}: {_html_path}')
except Exception as _e:
    print(f'HTML batch export failed: {_e}')
    import traceback; traceback.print_exc()

print("\n" + "="*60)


In [ ]:
# ==========================================================================
# NUMERICAL VERIFICATION: Fine mesh vs Simplified mesh overlap
#
# For each patch, verify that:
#   (a) Key vertex positions from patm.t/patm.p match spm.X
#   (b) Fine mesh centroid is near the simplified mesh face centroid
# ==========================================================================
import numpy as np
from pySHP.utils import kk_sph2cart, kk_cart2sph

spm = PM['spm']
pm  = PM['pm']
Xkeyind = PM['Xkeyind']
nkeys = len(PM.get('Keyind', []))

print("=" * 80)
print("VERIFICATION: Fine mesh vs Simplified mesh overlap per patch")
print("=" * 80)

for pix in range(PM['npatches']):
    patm = PM['P'][pix][0]
    print(f"\n--- Patch {pix} ---")
    
    # [A] Identify key vertices on this patch's boundary
    patch_face_idx = np.where(pm.face_labels == pix)[0] if pm.face_labels is not None else []
    patch_keys_simpl = set()
    for fi in patch_face_idx:
        for vi in spm.F[fi]:
            vi = int(vi)
            if vi < nkeys:
                patch_keys_simpl.add(vi)
    
    # [B] Compare key vertex sphere positions
    print(f"  Key vertices (simplified idx -> mX idx):")
    max_key_err = 0.0
    for si in sorted(patch_keys_simpl):
        mX_idx = int(Xkeyind[si])
        # Expected position from spm
        expected = spm.X[si]
        # Actual position from patm.t, patm.p
        if patm.t is not None and patm.p is not None and mX_idx < len(patm.t):
            actual_u, actual_v, actual_w = kk_sph2cart(
                np.array([patm.t[mX_idx]]), np.array([patm.p[mX_idx]]), np.array([1.0]))
            actual = np.array([actual_u[0], actual_v[0], actual_w[0]])
            err = np.linalg.norm(actual - expected)
            max_key_err = max(max_key_err, err)
            status = "OK" if err < 0.01 else f"** MISMATCH err={err:.4f} **"
            print(f"    v{si} (mX={mX_idx}): expected={expected.round(4)}, "
                  f"actual={actual.round(4)}, err={err:.6f} {status}")
            # Also check if t,p are near zero (not parameterized)
            if abs(patm.t[mX_idx]) < 1e-10 and abs(patm.p[mX_idx]) < 1e-10:
                print(f"      *** WARNING: t=0, p=0 - vertex NOT parameterized! ***")
        else:
            print(f"    v{si} (mX={mX_idx}): patm.t/p not available or vertex out of range")
    
    # [C] Compute fine mesh centroid on sphere
    if patm.t is not None and patm.p is not None:
        # Get vertices used by this patch's faces
        used_verts = np.unique(patm.F.flatten())
        t_used = patm.t[used_verts]
        p_used = patm.p[used_verts]
        u_fine, v_fine, w_fine = kk_sph2cart(t_used, p_used, np.ones(len(t_used)))
        fine_centroid = np.array([u_fine.mean(), v_fine.mean(), w_fine.mean()])
        fine_centroid_norm = np.linalg.norm(fine_centroid)
        if fine_centroid_norm > 1e-15:
            fine_centroid_hat = fine_centroid / fine_centroid_norm
        else:
            fine_centroid_hat = fine_centroid
        
        # [D] Compute simplified mesh face centroid on sphere
        if len(patch_face_idx) > 0:
            face_centroids = []
            for fi in patch_face_idx:
                verts = spm.X[spm.F[fi]]
                face_centroids.append(verts.mean(axis=0))
            simpl_centroid = np.mean(face_centroids, axis=0)
            simpl_centroid_norm = np.linalg.norm(simpl_centroid)
            if simpl_centroid_norm > 1e-15:
                simpl_centroid_hat = simpl_centroid / simpl_centroid_norm
            else:
                simpl_centroid_hat = simpl_centroid
            
            # Angular distance between centroids
            cos_angle = np.clip(np.dot(fine_centroid_hat, simpl_centroid_hat), -1, 1)
            angle_deg = np.degrees(np.arccos(cos_angle))
            
            status = "OK" if angle_deg < 15 else f"** OFFSET={angle_deg:.1f}deg **"
            print(f"  Fine mesh centroid:  {fine_centroid_hat.round(4)}")
            print(f"  Simpl mesh centroid: {simpl_centroid_hat.round(4)}")
            print(f"  Angular offset: {angle_deg:.2f} degrees {status}")
        
        # [E] Check if border vertex of patm is actually part of its faces
        border_verts = np.where(patm.border_vertex.astype(bool))[0] if hasattr(patm, 'border_vertex') and patm.border_vertex is not None else np.array([])
        face_verts_set = set(patm.F.flatten())
        border_in_faces = len(set(border_verts) & face_verts_set)
        border_not_in_faces = len(set(border_verts) - face_verts_set)
        if border_not_in_faces > 0:
            print(f"  ** WARNING: {border_not_in_faces} border vertices NOT in any face! **")
        
        # [F] Check how many face vertices have t=0, p=0 (unparameterized)
        n_unparam = 0
        for vi in used_verts:
            if abs(patm.t[vi]) < 1e-10 and abs(patm.p[vi]) < 1e-10:
                n_unparam += 1
        if n_unparam > 0:
            print(f"  ** WARNING: {n_unparam}/{len(used_verts)} face vertices have t=0, p=0 (not parameterized)! **")
        print(f"  Max key vertex error: {max_key_err:.6f}")

print("\n" + "=" * 80)
print("Verification complete.")
print("=" * 80)

In [ ]:
# ==========================================================================
# Diagnostic HTML: Per-patch view with simplified mesh overlay
#
# For each patch:
#   - ALL simplified mesh faces on the unit sphere (gray, wireframe)
#   - Faces belonging to THIS patch (green, solid)
#   - Key vertices on this patch boundary (red spheres, labelled)
#   - Center vertex of this patch (blue sphere)
#   - Other vertices (small gray dots for context)
# ==========================================================================
import os as _os
import numpy as np
import pyvista as pv
from pySHP.utils import kk_sph2cart

html_dir = _os.path.join(code_dir, 'pySHP', 'tests', 'patch_diag_html')
_os.makedirs(html_dir, exist_ok=True)

spm = PM['spm']            # simplified mesh on the unit sphere
pm  = PM['pm']              # simplified mesh (original coords)
Xkeyind = PM['Xkeyind']
nkeys = len(PM.get('Keyind', []))

# Build simplified mesh faces PyVista mesh
spm_nf = spm.F.shape[0]
spm_cells = np.hstack((np.full((spm_nf, 1), 3), spm.F)).flatten()
spm_mesh = pv.PolyData(spm.X.copy(), spm_cells)

for _pix in range(PM['npatches']):
    _p = pv.Plotter()

    # --- reference sphere ---
    _sphere = pv.Sphere(radius=0.97, theta_resolution=30, phi_resolution=30)
    _p.add_mesh(_sphere, color='cyan', opacity=0.08, show_edges=False)

    # --- ALL simplified mesh faces in gray wireframe ---
    _p.add_mesh(spm_mesh.copy(), color='lightgray', style='wireframe',
                line_width=2, opacity=0.5)

    # --- This patch's simplified mesh faces in GREEN ---
    patch_face_idx = np.where(pm.face_labels == _pix)[0] if pm.face_labels is not None else np.array([])
    if len(patch_face_idx) > 0:
        patch_faces = spm.F[patch_face_idx]
        pf_cells = np.hstack((np.full((len(patch_faces), 1), 3), patch_faces)).flatten()
        patch_mesh = pv.PolyData(spm.X.copy(), pf_cells)
        _p.add_mesh(patch_mesh, color='green', show_edges=True,
                    edge_color='darkgreen', line_width=3, opacity=0.6)

    # --- Collect this patch's key vertices (boundary) ---
    # From the diagnostic: keys per patch
    patch_keys_simpl = set()
    for fi in patch_face_idx:
        for vi in spm.F[fi]:
            vi = int(vi)
            if vi < nkeys:  # it's a key vertex
                patch_keys_simpl.add(vi)

    # --- Center vertex of this patch (its fan-center) ---
    patch_center_simpl = set()
    for fi in patch_face_idx:
        for vi in spm.F[fi]:
            vi = int(vi)
            if vi >= nkeys:  # center/fictitious vertex
                patch_center_simpl.add(vi)

    # --- Key vertices on this patch boundary -> RED ---
    if len(patch_keys_simpl) > 0:
        key_pts = np.array([spm.X[si] for si in sorted(patch_keys_simpl)])
        key_labels = [f'v{si} (mX={int(Xkeyind[si])})' for si in sorted(patch_keys_simpl)]
        key_cloud = pv.PolyData(key_pts)
        _p.add_mesh(key_cloud, color='red', point_size=14,
                    render_points_as_spheres=True, label='Key vertices')
        # Add labels
        _p.add_point_labels(key_cloud, key_labels, font_size=10,
                           text_color='red', point_size=0, shape=None,
                           render_points_as_spheres=False, always_visible=True)

    # --- Center vertex of this patch -> BLUE ---
    if len(patch_center_simpl) > 0:
        ctr_pts = np.array([spm.X[si] for si in sorted(patch_center_simpl)])
        ctr_labels = [f'c{si} (mX={int(Xkeyind[si])})' for si in sorted(patch_center_simpl)]
        ctr_cloud = pv.PolyData(ctr_pts)
        _p.add_mesh(ctr_cloud, color='blue', point_size=14,
                    render_points_as_spheres=True, label='Center vertex')
        _p.add_point_labels(ctr_cloud, ctr_labels, font_size=10,
                           text_color='blue', point_size=0, shape=None,
                           render_points_as_spheres=False, always_visible=True)

    # --- Other key vertices (not on this patch) -> small gray ---
    other_keys = set(range(nkeys)) - patch_keys_simpl
    if len(other_keys) > 0:
        other_pts = np.array([spm.X[si] for si in sorted(other_keys)])
        _p.add_mesh(pv.PolyData(other_pts), color='gray', point_size=5,
                    render_points_as_spheres=True, opacity=0.5)

    # --- Fine-mesh patch on sphere (if parameterized) ---
    _patm = PM['P'][_pix][0]
    if _patm.t is not None and _patm.p is not None and np.any(_patm.t != 0):
        _u, _v, _w = kk_sph2cart(_patm.t, _patm.p, np.ones(len(_patm.p)))
        _Xsph = np.column_stack([_u, _v, _w])
        _nf = _patm.F.shape[0]
        _cells = np.hstack((np.full((_nf, 1), 3), _patm.F)).flatten()
        _mesh_sph = pv.PolyData(_Xsph, _cells)
        _p.add_mesh(_mesh_sph, color='lightyellow', show_edges=True,
                    edge_color='black', line_width=0.3, opacity=0.35)

        # Mark border vertices of fine mesh on sphere
        if hasattr(_patm, 'border_vertex') and _patm.border_vertex is not None:
            _bv = np.where(_patm.border_vertex.astype(bool))[0]
            _bv = _bv[(_patm.t[_bv] != 0) | (_patm.p[_bv] != 0)]
            if len(_bv) > 0:
                _p.add_mesh(pv.PolyData(_Xsph[_bv]), color='yellow',
                            point_size=4, render_points_as_spheres=True,
                            label='Fine mesh border')

    # --- Edge info annotation ---
    edge_info_lines = [f'Patch {_pix}: {len(patch_face_idx)} simpl faces, '
                       f'{_patm.F.shape[0]} fine faces']
    edge_info_lines.append(f'Keys: {sorted(patch_keys_simpl)} -> mX={[int(Xkeyind[s]) for s in sorted(patch_keys_simpl)]}')
    if len(patch_center_simpl) > 0:
        edge_info_lines.append(f'Center: {sorted(patch_center_simpl)} -> mX={[int(Xkeyind[s]) for s in sorted(patch_center_simpl)]}')
    # Edge chains
    for eix in range(len(PM['Edges'])):
        if PM['Edges'][eix, 0] == _pix or PM['Edges'][eix, 1] == _pix:
            s1, s2 = int(PM['sentinels'][eix, 0]), int(PM['sentinels'][eix, 1])
            chain_len = len(PM['edge_dat'][eix]) if eix < len(PM['edge_dat']) else 0
            neighbor = int(PM['Edges'][eix, 1]) if PM['Edges'][eix, 0] == _pix else int(PM['Edges'][eix, 0])
            edge_info_lines.append(f'  eix={eix}: ({s1},{s2}) len={chain_len} nb={neighbor}')

    _p.add_text('\n'.join(edge_info_lines), font_size=8, position='lower_left')
    _p.background_color = 'white'

    _html_path = _os.path.join(html_dir, f'patch_{_pix}_diag.html')
    _p.export_html(_html_path)
    _p.close()
    print(f'  Patch {_pix}: {_html_path}')

print('\nDone. Open the HTML files in a browser to inspect each patch.')
print(f'Directory: {html_dir}')

### Assemble full parameterized mesh from all patches

In [ ]:
# This combines the t and p values from all parameterized patches into the full mesh
print("="*60)
print("Assembling full parameterized mesh from all patches")
print("="*60)

# Use m_seg (the segmented mesh) as the base for the full parameterized mesh
# m_seg has the same vertices and faces as the original mesh, with face_labels
mp = surface_mesh(m_seg.X.copy(), m_seg.F.copy())
mp.face_labels = m_seg.face_labels.copy() if hasattr(m_seg, 'face_labels') and m_seg.face_labels is not None else None

# Initialize t and p arrays
mp.t = np.zeros(len(mp.X))
mp.p = np.zeros(len(mp.X))

# Copy t and p values from each parameterized patch to the full mesh
for pix in range(PM['npatches']):
    pat = PM['P'][pix][0]  # Get the parameterized patch
    # Get all vertex indices used in this patch
    Vix = np.unique(pat.F.flatten())
    
    # Copy t and p values from patch to full mesh
    for v in Vix:
        if v < len(mp.t) and v < len(pat.t) and pat.t[v] != 0:
            mp.t[v] = pat.t[v]
        if v < len(mp.p) and v < len(pat.p) and pat.p[v] != 0:
            mp.p[v] = pat.p[v]

print(f"Assembled full parameterized mesh:")
print(f"  Total vertices: {len(mp.X)}")
print(f"  Vertices with t values: {np.sum(mp.t != 0)}")
print(f"  Vertices with p values: {np.sum(mp.p != 0)}")
print(f"  Theta range: [{mp.t[mp.t != 0].min():.4f}, {mp.t[mp.t != 0].max():.4f}]")
print(f"  Phi range: [{mp.p[mp.p != 0].min():.4f}, {mp.p[mp.p != 0].max():.4f}]")

# Fix and check spherical normals (inward-facing faces can occur when merging patches)
from pySHP.level1.fix_flipped_faces import check_spherical_parameterization_normals, fix_spherical_parameterization_normals
mp, _n = fix_spherical_parameterization_normals(mp, verbose=True)
_ = check_spherical_parameterization_normals(mp, verbose=True)

print("\n" + "="*60)

### Plot starting full parameterized mesh on the sphere

In [ ]:
# This shows all patches together on the sphere, similar to the MATLAB plot_spherical_parameterization(mp)
# Ensure PyVista backend is set so the figure displays (use 'static' to avoid empty figures)
import pyvista as pv
import os
pv.set_jupyter_backend('static')

print("="*60)
print("Plotting full parameterized mesh (all patches combined)")
print("="*60)

# Output directory for HTML (same as simplified patch HTMLs); open in browser if plot does not appear
base = code_dir if 'code_dir' in dir() else os.getcwd()
html_dir = os.path.join(base, 'pySHP', 'tests', 'simplified_patch_html')
os.makedirs(html_dir, exist_ok=True)
full_sphere_html = os.path.join(html_dir, 'full_parameterization_on_sphere.html')

# Plot the full mesh spherical parameterization and export to HTML for browser viewing
mp.plot_spherical_parameterization(flag=0, export_html_path=full_sphere_html)  # flag=1 for reference sphere
print("Full parameterized mesh plotted on sphere")
print(f"Open in browser: {full_sphere_html}")
print("\n" + "="*60)

## Final parameterization Optimization: Area/Shear and Shear

### Area/Shear optimization on sphere

In [ ]:
print("="*60)
print("Spherical Harmonics Projection and Export")
print("="*60)

# Step 1: Assemble full parameterized mesh from all patches
from pySHP.level1.assemble_parameterized_mesh import assemble_parameterized_mesh

print("\n[Step 1] Assembling full parameterized mesh from all patches...")
m_param = assemble_parameterized_mesh(m_seg, PM)
m_param.optimization_method = 2  # Use alternating area/shear correction
m_param.prevent_flip = True
m_param.newton_niter = 100        # Number of iterations
m_param.newton_step = 0.2        # Step factor
m_param.map2sphere()
m_param.needs_map2sphere = 0
# Fix and check spherical normals after re-optimization
from pySHP.level1.fix_flipped_faces import check_spherical_parameterization_normals, fix_spherical_parameterization_normals
m_param, _ = fix_spherical_parameterization_normals(m_param, verbose=True)
_ = check_spherical_parameterization_normals(m_param, verbose=True)


### Shear optimization on sphere

In [ ]:
m_param.optimization_method = 5  # Use shear correction
m_param.newton_niter = 100        # Number of iterations
m_param.newton_step = 0.02        # Step factor
m_param.prevent_flip = True
m_param.map2sphere()
m_param.needs_map2sphere = 0
# Fix and check spherical normals after shear correction
from pySHP.level1.fix_flipped_faces import fix_spherical_parameterization_normals, check_spherical_parameterization_normals
m_param, _ = fix_spherical_parameterization_normals(m_param, verbose=True)
_ = check_spherical_parameterization_normals(m_param, verbose=True)



### Export spherical mesh as html

In [ ]:
# For the assembled full fine mesh (after parameterization + assemble)
mp.html_mesh_parameterization_export(
    'pySHP/tests/simplified_patch_html/full_parameterization_on_sphere.html',
    show_reference_sphere=False,
    face_field=m_param.face_labels,  # optional: color by patch
    title='Full fine-mesh spherical parameterization'
)

## Spherical harmonics projection

In [ ]:
L_max: int = 12
s = shp_surface(m_param, L_max)
#s.plot(nico=3)
print(f"  Spherical harmonics surface created:")
print(f"    L_max: {s.L_max}")
print(f"    Number of coefficients: {len(s.xc)}")
print(f"    Residual (reconstruction error): {np.linalg.norm(s.residual):.6e}")

# Export to .shp3 file
print("\n[Step 3] Exporting to .shp3 file...")
output_filename = "parameterized_mesh.shp3" 
s.export_shp3(output_filename)

print("\n" + "="*60)
print("Spherical harmonics projection complete!")
print(f"  Output file: {output_filename}")
print("="*60)

# Export the shp3-derived mesh as off
output_filename = "parameterized_shp3_mesh.off"  # Change this to your desired filename
mshp3 = s.get_mesh(5)
mshp3.write_off(output_filename_shp3_mesh)

# Optional: Visualize the spherical harmonics reconstruction
print("\n[Optional] Visualizing spherical harmonics reconstruction...")
try:
    import pyvista as pv
    pv.set_jupyter_backend('static')
    # Get mesh from spherical harmonics
    m_shp, X_shp, F_shp, Y_LK, t_shp, p_shp = s.get_mesh(nico=3)
    
    # Create plotter
    plotter = pv.Plotter(shape=(1, 2))
    
    # Plot 1: Original parameterized mesh on sphere
    plotter.subplot(0, 0)
    # Convert t, p to Cartesian on unit sphere
    from pySHP.utils import kk_sph2cart
    u, v, w = kk_sph2cart(m_param.t, m_param.p, np.ones(len(m_param.p)))
    X_sph_orig = np.column_stack([u, v, w])
    
    # Create mesh
    num_faces_orig = m_param.F.shape[0]
    faces_with_n_vertices_orig = np.hstack((np.full((num_faces_orig, 1), 3), m_param.F))
    cells_orig = faces_with_n_vertices_orig.flatten()
    mesh_orig = pv.PolyData(X_sph_orig, cells_orig)
    plotter.add_mesh(mesh_orig, color='lightblue', show_edges=True, 
                    edge_color='black', line_width=0.5, opacity=1.0)
    plotter.add_text('Original Parameterized Mesh\n(on unit sphere)', font_size=10)
    plotter.background_color = 'black'
    
    # Plot 2: Spherical harmonics reconstruction
    plotter.subplot(0, 1)
    # Create mesh from SH coefficients
    num_faces_shp = F_shp.shape[0]
    faces_with_n_vertices_shp = np.hstack((np.full((num_faces_shp, 1), 3), F_shp))
    cells_shp = faces_with_n_vertices_shp.flatten()
    mesh_shp = pv.PolyData(X_shp, cells_shp)
    plotter.add_mesh(mesh_shp, color='lightgreen', show_edges=True, 
                    edge_color='black', line_width=0.5, opacity=1.0)
    plotter.add_text('Spherical Harmonics Reconstruction\n(L_max={})'.format(L_max), font_size=10)
    plotter.background_color = 'black'
    
    plotter.show()
    
except ImportError:
    print("PyVista not available for visualization")
except Exception as e:
    print(f"Visualization error: {e}")
    import traceback
    traceback.print_exc()
s.plot(nico = 5)
m_param.plot()
# quantify shear
shear_sph, summary_sph = m_param.compute_shear_spherical_mesh()
print("Shear (spherical):", summary_sph['mean'])
m_param.plot_shear_heatmap_spherical_mesh(title='Shear on sphere (fine)')
m_param.plot_shear_heatmap_3d_mesh(title='Shear on 3D (fine)')